In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "NeoIsTheOne"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load the Grazioso Salvare logo from notebook file file.
image_filename = "Grazioso Salvare Logo.png"

# Read the image as binary data, then encode it so Dash can display it.
with open(image_filename, "rb") as image_file:
    encoded_image = base64.b64encode(image_file.read()).decode("utf-8")

# Create an image component for the dashboard.
logo = html.A(
    html.Img(
        src=f"data:image/png;base64,{encoded_image}",
        style={
            "height": "100px",
            "width": "auto"
        }
    ),
    href="https://www.snhu.edu",
    target="_blank"
)

# Build the dashboard layout.
app.layout = html.Div([
    # Display the logo and a unique identifier at the top of the dashboard.
    html.Div(
        [
            logo,
            html.P(
                "Benjamin Holcomb — CS-340",  # Replaced with my name
                style={"marginTop": "8px"}
            )
        ],
        style={
            "textAlign": "center",
            "padding": "10px"
        }
    ),

    # Dashboard title.
    html.Center(html.B(html.H1("CS-340 Dashboard"))),
    html.Hr(),
    # Interactive controls used to choose which rescue-dog profile to show.
    html.Div([
        dcc.RadioItems(
            id="filter-type",  # The callback below listens to this component.
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"},
                {"label": "Reset — show all animals", "value": "reset"},
            ],
            value="reset",  # Start with all animals displayed.
            labelStyle={
                "display": "inline-block",
                "marginRight": "20px"
            },
        )
    ]),
    html.Hr(),
    # Display the animal records retrieved by the unfiltered query above.
    dash_table.DataTable(
        id="datatable-id",

        # Use the DataFrame's columns as the table's column definitions.
        columns=[
            {
                "name": column,
                "id": column,
                "deletable": False,
                "selectable": True
            }
            for column in df.columns
        ],

        # Show all records initially. The filter callback will update these rows.
        data=df.to_dict("records"),

        # Make the table easier to navigate and inspect.
        page_action="native",
        page_size=10,
        sort_action="native",
        filter_action="none",

        # Allow one row to be selected for the geolocation map.
        row_selectable="single",
        selected_rows=[0],

        # Improve readability.
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "padding": "8px",
            "minWidth": "100px",
            "maxWidth": "250px",
            "whiteSpace": "normal",
            "height": "auto"
        },
        style_header={
            "fontWeight": "bold",
            "backgroundColor": "#e8f0f5"
        }
    ),                 
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
# Update the table whenever the user chooses a rescue type.
@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_dashboard(filter_type):

    # Each query matches a rescue profile from the specifications document.
    # "$in" means the dog's breed can match any breed in the list.
    # "$gte" and "$lte" set the inclusive minimum and maximum age.
    filters = {
        "water": {
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        },

        "mountain": {
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        },

        "disaster": {
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }
    }

    # For "reset", use an empty query, which asks MongoDB for all documents.
    # Otherwise, use the query associated with the selected radio-button value.
    query = filters.get(filter_type, {})

    # Ask the CRUD module to retrieve matching documents from MongoDB.
    records = db.read(query)

    # Remove MongoDB's ObjectId before passing records to the Dash table.
    for record in records:
        record.pop("_id", None)

    # The DataTable's "data" property expects a list of dictionaries.
    return records
# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])

def update_graphs(viewData):
    # Show an empty-state chart if the table has no rows.
    if not viewData:
        return dcc.Graph(
            figure=px.pie(title="No animals match this filter")
        )

    # Convert the rows currently shown in the table into a DataFrame.
    dff = pd.DataFrame.from_records(viewData)

    # Count animals per breed, from most common to least common.
    counts = dff["breed"].value_counts()

    # Keep the 10 most common breeds as individual pie slices.
    top_breeds = counts.head(10)

    # Add together the counts for all remaining breeds.
    other_count = counts.iloc[10:].sum()

    # Format the top breeds for Plotly Express.
    breed_counts = top_breeds.reset_index()
    breed_counts.columns = ["breed", "count"]

    # Add one "Other" slice if any breeds were left out of the top 10.
    if other_count > 0:
        breed_counts.loc[len(breed_counts)] = ["Other", other_count]

    # Build the pie chart using the updated table data.
    return dcc.Graph(
        figure=px.pie(
            breed_counts,
            names="breed",
            values="count",
            title="Animals by Breed"
        )
)
#This callback will highlight a cell on the data table when the user selects it
# Highlight any columns the user has selected in the data table.
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def update_styles(selected_columns):
    # Dash may pass None when no columns are selected.
    # Use an empty list so the loop below can run safely.
    selected_columns = selected_columns or []

    # Return one conditional style rule for each selected column.
    return [
        {
            "if": {"column_id": column},
            "backgroundColor": "#D2F3FF"
        }
        for column in selected_columns
    ]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):
    # Show a message if the table has no rows.
    if not viewData:
        return html.Div("No animals to display on the map.")

    # Convert the table rows into a DataFrame.
    dff = pd.DataFrame.from_records(viewData)

    # Use the selected row, or the first row if none is selected.
    row = index[0] if index else 0
    if row >= len(dff):
        row = 0

    # Get the selected animal's details.
    animal = dff.iloc[row]
    name = animal["name"]
    if pd.isna(name) or not name:
        name = "Name not recorded"

    # Center the map and marker on this animal's coordinates.
    latitude = animal["location_lat"]
    longitude = animal["location_long"]

    return dl.Map(
        style={"width": "400px", "height": "500px"},
        center=[latitude, longitude],
        zoom=10,
        children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(animal["breed"]),
                    dl.Popup([
                        html.H1(name),
                        html.P(f"Breed: {animal['breed']}")
                    ])
                ]
            )
        ]
    )
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://bookringo-raymondlocate-3000.codio.io/proxy/8050/
